# Predicción del reemplazo de baterias de litio - SVM 

---

**Autor:** Borja Mora Mendez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoria:** Machine Learning · Supervisado · Regresion · Support Vector Machines

---

### Objetivo

Predecir la **capacidad restante (%)** de una bateria a partir de su edad e intensidad de uso, usando SVR con diferentes kernels. La degradacion de baterias sigue un patron no lineal (exponencial/sigmoide) — un caso ideal para SVR.

**El cliente:** un fabricante de dispositivos electronicos que necesita predecir cuando sus baterias necesitaran reemplazo.

**El problema:** reemplazar una bateria demasiado pronto es un coste innecesario; demasiado tarde causa fallos en campo. Un modelo que prediga la capacidad restante permite programar reemplazos preventivos.

**La pregunta:** se puede predecir la capacidad de una bateria (%) a partir de su edad y patron de uso?


- **Edad_Anos**: cuántos años tiene la batería
- **Intensidad_Uso**: qué tan intensamente se ha usado (0 a 100)
- **Requiere_Reemplazo**: la variable objetivo → 0 (sana) o 1 (necesita reemplazo)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.lines import Line2D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Estilo visual
PURPLE = '#7a7bff'
GREEN  = '#6E7F5B'
RED    = '#C2412E'
GRAY   = '#F4EFE6'
INK    = '#2B2118'
MUTED  = '#bfbfbf'

plt.rcParams.update({
    'figure.figsize': (10, 5), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': MUTED, 'axes.labelcolor': INK,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.titlecolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'font.family': 'sans-serif', 'font.size': 10,
    'grid.color': '#f0f0f0', 'grid.linewidth': 0.5
})


## 1. Carga y validación de datos

In [ ]:
data = pd.read_excel("datos_baterias.xlsx")
print(f"Registros: {data.shape[0]} | Columnas: {data.shape[1]}")
data.head()


## 2. EDA

In [ ]:
display(data.describe())

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=data, x="Edad_Anos", y="Intensidad_Uso", hue="Requiere_Reemplazo", palette={0:GREEN,1:RED}, s=75, edgecolor="black", alpha=.8, ax=ax)
ax.set_title("Distribución de baterías según estado")
ax.set_xlabel("Edad de la batería (años)")
ax.set_ylabel("Intensidad de uso")
ax.legend(title="Estado", labels=["Sana (0)", "Reemplazo (1)"])
ax.grid(True, linestyle=":", alpha=.6)
plt.tight_layout(); plt.show()


## 3. Preparación: Train / Test y estandarización

In [ ]:
X = data[["Edad_Anos", "Intensidad_Uso"]]
y = data["Requiere_Reemplazo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

escalador = StandardScaler()
X_train_scaled = escalador.fit_transform(X_train)
X_test_scaled = escalador.transform(X_test)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")


## 4. SVM — Kernel lineal

In [ ]:
modelo_lineal = SVC(kernel="linear", C=2.0, random_state=42)
modelo_lineal.fit(X_train_scaled, y_train)
y_pred_lineal = modelo_lineal.predict(X_test_scaled)

print(classification_report(y_test, y_pred_lineal, target_names=["Está Sana (0)", "Necesita Reemplazo (1)"], digits=3))


## 5. SVM — Kernel polinómico grado 2

In [ ]:
modelo_poly2 = SVC(kernel="poly", degree=2, C=1.0, gamma="scale", coef0=1, random_state=42)
modelo_poly2.fit(X_train_scaled, y_train)
y_pred_poly2 = modelo_poly2.predict(X_test_scaled)

print(classification_report(y_test, y_pred_poly2, target_names=["Está Sana (0)", "Necesita Reemplazo (1)"], digits=3))


## 6. SVM — Kernel polinómico grado 3

In [ ]:
modelo_poly3 = SVC(kernel="poly", degree=3, C=1.0, gamma="scale", coef0=1, random_state=42)
modelo_poly3.fit(X_train_scaled, y_train)
y_pred_poly3 = modelo_poly3.predict(X_test_scaled)

print(classification_report(y_test, y_pred_poly3, target_names=["Está Sana (0)", "Necesita Reemplazo (1)"], digits=3))


## 7. Comparación visual de las fronteras

In [ ]:
x_min, x_max = data["Edad_Anos"].min()-0.5, data["Edad_Anos"].max()+0.5
y_min, y_max = data["Intensidad_Uso"].min()-5, data["Intensidad_Uso"].max()+5
xx, yy = np.meshgrid(np.linspace(x_min,x_max,500), np.linspace(y_min,y_max,500))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = escalador.transform(grid)

modelos = [("SVM Lineal", modelo_lineal), ("SVM Poly grado 2", modelo_poly2), ("SVM Poly grado 3", modelo_poly3)]
fig, axes = plt.subplots(1,3,figsize=(18,5),sharey=True)

for ax,(titulo,modelo) in zip(axes,modelos):
    decision = modelo.decision_function(grid_scaled).reshape(xx.shape)
    sns.scatterplot(data=data,x="Edad_Anos",y="Intensidad_Uso",hue="Requiere_Reemplazo",palette={0:GREEN,1:RED},s=55,edgecolor="black",linewidth=.4,alpha=.8,legend=False,ax=ax)
    ax.contour(xx,yy,decision,levels=[0],colors=INK,linewidths=2.5)
    ax.set_title(titulo)
    ax.set_xlabel("Edad (años)")
    ax.grid(True,linestyle=":",alpha=.6)
axes[0].set_ylabel("Intensidad de uso")
handles=[Line2D([0],[0],marker='o',color='none',markerfacecolor=GREEN,markeredgecolor='black',markersize=8,label='Sana (0)'),Line2D([0],[0],marker='o',color='none',markerfacecolor=RED,markeredgecolor='black',markersize=8,label='Reemplazo (1)'),Line2D([0],[0],color=INK,lw=2.5,label='Frontera de decisión')]
fig.legend(handles=handles,loc='lower center',ncol=3,frameon=False,bbox_to_anchor=(.5,-.04))
fig.suptitle("Comparación de fronteras de decisión SVM",fontweight='bold',color=INK)
plt.tight_layout(); plt.show()


## 8. Comparación final

In [ ]:
modelos = {
    "SVM Lineal": (modelo_lineal, y_pred_lineal),
    "SVM Poly grado 2": (modelo_poly2, y_pred_poly2),
    "SVM Poly grado 3": (modelo_poly3, y_pred_poly3),
}

filas=[]
for nombre,(modelo,pred) in modelos.items():
    r=classification_report(y_test,pred,output_dict=True)
    filas.append({
        "Modelo":nombre,
        "Precision Reemplazo":round(r["1"]["precision"],3),
        "Recall Reemplazo":round(r["1"]["recall"],3),
        "F1 Reemplazo":round(r["1"]["f1-score"],3),
        "Accuracy":round(r["accuracy"],3),
        "F1 Macro":round(r["macro avg"]["f1-score"],3),
        "Vectores Soporte":len(modelo.support_vectors_)
    })

df_comparativa=pd.DataFrame(filas).sort_values("Recall Reemplazo",ascending=False)
display(df_comparativa)


## 9. Conclusión final

El ejercicio confirma que **SVM es capaz de separar las baterías según su necesidad de reemplazo utilizando únicamente dos variables: edad e intensidad de uso**.

Con el dataset corregido de **119 registros**, el test contiene 36 observaciones. Al volver a ejecutar el notebook, los resultados obtenidos son:

- **SVM Lineal:** accuracy **88,9%** y recall de reemplazo **92,3%**.
- **SVM Poly grado 2:** accuracy **91,7%** y recall de reemplazo **96,2%**.
- **SVM Poly grado 3:** accuracy **91,7%** y recall de reemplazo **96,2%**.

### Lectura de negocio

El criterio más importante no debería ser únicamente la accuracy. En este caso, es especialmente relevante el **recall de la clase 1**, porque representa la capacidad del modelo para detectar baterías que realmente necesitan reemplazo.

Por tanto, **Poly grado 2 y Poly grado 3 son las mejores alternativas de las tres probadas**: detectan aproximadamente el **96% de las baterías que requieren reemplazo** y alcanzan un **91,7% de accuracy** en el conjunto de test. El grado 3 no aporta una mejora frente al grado 2, por lo que **Poly grado 2 sería la opción preferible por simplicidad**.

Hay, no obstante, una limitación importante: el dataset es pequeño y el conjunto de test contiene solo 36 observaciones. Por ello, estos resultados deben interpretarse como una primera prueba del enfoque y no como evidencia suficiente para desplegar el modelo en producción. El siguiente paso sería validar los kernels mediante **validación cruzada** y ampliar el dataset con más baterías reales.
